# V4.2 Phase 3 Step 2 — Train the Neural Net

Trains on `features_v4.parquet` — 106k snapshots from 5 leagues, 3 seasons,
built using Dixon-Coles strength features instead of hand-curated FIFA ranks.

Same architecture as Phase 2b (11→40→20→3). Same two-stage training. Same
sqrt-inverse class weights that fixed the draw-spam problem. Same temperature
calibration. Output goes to `v4_backend/models/` so nothing overwrites the
working World Cup model.


## Cell 1 — Setup

In [ ]:
import pickle, warnings
from datetime import datetime, timezone
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, log_loss,
                             confusion_matrix, classification_report,
                             precision_score, recall_score)
warnings.filterwarnings('ignore')

ROOT       = Path('.')
DATA_PROC  = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'v4_backend' / 'models'
RESULTS    = ROOT / 'results'
for d in (MODELS_DIR, RESULTS): d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)


## Cell 2 — Load data

In [ ]:
FEATURE_COLS = [
    'goal_diff', 'minute_norm', 'is_second_half',
    'home_rank_norm', 'away_rank_norm', 'rank_diff',
    'is_knockout', 'lead_changes_norm',
    'is_neutral_venue', 'score_state', 'strength_x_time',
]
TARGET_COL = 'outcome'
N_FEATURES, N_CLASSES = len(FEATURE_COLS), 3

df = pd.read_parquet(DATA_PROC / 'features_v4.parquet')
X      = df[FEATURE_COLS].values.astype('float32')
y      = df[TARGET_COL].values.astype('int64')
groups = df['match_id'].values

print(f'Loaded {len(df):,} snapshots, {N_FEATURES} features')
print(f'Outcome split -- home {(y==2).mean():.1%}  draw {(y==1).mean():.1%}  away {(y==0).mean():.1%}')


## Cell 3 — Split by match (no leakage)

All snapshots from one match must stay together -- the same leakage rule
from the World Cup model. 60% train / 20% val / 20% test.


In [ ]:
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx, test_idx = next(gss1.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_rel, va_rel = next(gss2.split(
    X[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]
val_idx   = trainval_idx[va_rel]

# Proof: no match appears in two piles
tr_m = set(groups[train_idx])
va_m = set(groups[val_idx])
te_m = set(groups[test_idx])
assert not (tr_m & va_m) and not (tr_m & te_m) and not (va_m & te_m)
print('Leakage check passed -- no match spans two splits.')

scaler = StandardScaler().fit(X[train_idx])
Xtr = scaler.transform(X[train_idx]).astype('float32')
Xva = scaler.transform(X[val_idx]).astype('float32')
Xte = scaler.transform(X[test_idx]).astype('float32')
ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]

def make_loader(Xa, ya, batch=256, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya)),
                      batch_size=batch, shuffle=shuffle)

train_loader = make_loader(Xtr, ytr, shuffle=True)
val_loader   = make_loader(Xva, yva)

print(f'train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,} snapshots')
print(f'train {len(tr_m):,} | val {len(va_m):,} | test {len(te_m):,} unique matches')


## Cell 4 — Class weights (sqrt-inverse, same fix as Phase 2b)

Prevents the model spamming one outcome. Draw gets the largest weight
since it's the smallest class and the hardest to predict correctly.


In [ ]:
counts = np.bincount(ytr, minlength=3).astype('float64')
freqs  = counts / counts.sum()
w_sqrt = 1.0 / np.sqrt(freqs)
w_sqrt = w_sqrt / w_sqrt.sum() * 3.0
class_weights = torch.tensor(w_sqrt, dtype=torch.float32, device=DEVICE)

print('Class       away    draw    home')
print('Frequency  ', '  '.join(f'{f:6.1%}' for f in freqs))
print('Weight     ', '  '.join(f'{w:6.2f}' for w in w_sqrt))
print()
print('Draw has the largest weight -- trained to not ignore draws.')


## Cell 5 — Build the network (11→40→20→3)

Same architecture as Phase 2b. Small enough to generalise across 106k
snapshots without memorising them, large enough to find real patterns.


In [ ]:
class FootballWinProbNet(nn.Module):
    def __init__(self, n_features=11, n_classes=3, h1=40, h2=20, dropout=0.30):
        super().__init__()
        self.fc1  = nn.Linear(n_features, h1)
        self.fc2  = nn.Linear(h1, h2)
        self.head = nn.Linear(h2, n_classes)
        self.drop = nn.Dropout(dropout)
        self.act  = nn.ReLU()

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        x = self.drop(self.act(self.fc2(x)))
        return self.head(x)

model = FootballWinProbNet(N_FEATURES, N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'Total parameters: {n_params:,}')

# Warm-start: try NBA weights for general game-sense, fall through gracefully
nba_candidates = [
    Path('notebooks/models/football_v2.pth'),
    Path('models/football_v2.pth'),
]
nba_path = next((p for p in nba_candidates if p.exists()), None)
TRANSFER_DONE = False
if nba_path:
    ckpt = torch.load(nba_path, map_location='cpu', weights_only=False)
    state = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt))
    first_layer = next((v for v in state.values()
                        if hasattr(v, 'ndim') and v.ndim == 2 and v.shape[1] >= 7), None)
    if first_layer is not None:
        r = min(model.fc1.weight.shape[0], first_layer.shape[0])
        c = min(model.fc1.weight.shape[1], first_layer.shape[1])
        with torch.no_grad():
            model.fc1.weight.data[:r, :c] = first_layer[:r, :c].float()
        print(f'Warm-started fc1 from {nba_path} ({r}x{c} slice)')
        TRANSFER_DONE = True
else:
    print('No prior model found -- training from random init (still fine).')


## Cell 6 — Two-stage training

Stage 1 (12 epochs): freeze the warm-started first layer, train only
the new layers. Lets the new football-specific layers learn before
updating the general game-sense in fc1.

Stage 2 (up to 40 epochs with early stopping): unfreeze everything,
fine-tune gently at 10x lower learning rate.


In [ ]:
def run_epoch(net, loader, optimizer=None):
    net.train() if optimizer else net.eval()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    total, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = net(xb)
            loss   = crit(logits, yb)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total   += loss.item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
            n       += len(xb)
    return total / n, correct / n

# Stage 1
model.fc1.weight.requires_grad = False
model.fc1.bias.requires_grad   = False
opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=3e-3)
print('Stage 1 (fc1 frozen):')
for epoch in range(1, 13):
    tl, ta = run_epoch(model, train_loader, opt1)
    vl, va = run_epoch(model, val_loader)
    if epoch % 3 == 0 or epoch == 1:
        print(f'  epoch {epoch:2d}  train_acc {ta:.3f}  val_acc {va:.3f}')

# Stage 2
for p in model.parameters(): p.requires_grad = True
opt2 = torch.optim.Adam(model.parameters(), lr=3e-4)
best_val, best_state, patience, bad = 1e9, None, 6, 0
print('Stage 2 (full fine-tune):')
for epoch in range(1, 41):
    tl, ta = run_epoch(model, train_loader, opt2)
    vl, va = run_epoch(model, val_loader)
    if vl < best_val - 1e-4:
        best_val, best_state, bad = vl, deepcopy(model.state_dict()), 0
        star = ' *'
    else:
        bad += 1; star = ''
    if epoch % 4 == 0 or epoch == 1 or star:
        print(f'  epoch {epoch:2d}  train_acc {ta:.3f}  val_acc {va:.3f}  val_loss {vl:.4f}{star}')
    if bad >= patience:
        print(f'  Early stop at epoch {epoch}'); break

model.load_state_dict(best_state)
print('Training done -- restored best-validation weights.')


## Cell 7 — Temperature calibration

In [ ]:
model.eval()
with torch.no_grad():
    val_logits = model(torch.tensor(Xva).to(DEVICE)).cpu().numpy()

def nll_at_T(T):
    z = val_logits / T
    z = z - z.max(1, keepdims=True)
    p = np.exp(z); p = p / p.sum(1, keepdims=True)
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return log_loss(yva, p, labels=[0, 1, 2])

Ts = np.linspace(0.5, 3.0, 60)
T_best = float(Ts[int(np.argmin([nll_at_T(T) for T in Ts]))])
print(f'Best temperature T = {T_best:.3f}')
print(f'  NLL before: {nll_at_T(1.0):.4f}')
print(f'  NLL after : {nll_at_T(T_best):.4f}')


## Cell 8 — Grade on the locked test set

In [ ]:
def probs_at_T(logits, T):
    z = logits / T; z = z - z.max(1, keepdims=True)
    p = np.exp(z); return p / p.sum(1, keepdims=True)

with torch.no_grad():
    test_logits = model(torch.tensor(Xte).to(DEVICE)).cpu().numpy()

test_p    = probs_at_T(test_logits, T_best)
test_pred = test_p.argmax(1)
names     = ['away', 'draw', 'home']

overall_acc  = accuracy_score(yte, test_pred)
baseline_acc = max(np.bincount(yte)) / len(yte)
ll           = log_loss(yte, test_p, labels=[0, 1, 2])
cm           = confusion_matrix(yte, test_pred, labels=[0, 1, 2])
draw_recall  = recall_score(yte, test_pred, labels=[1], average='macro', zero_division=0)
draw_prec    = precision_score(yte, test_pred, labels=[1], average='macro', zero_division=0)

print(f'Overall accuracy : {overall_acc:.3f}')
print(f'Baseline (home)  : {baseline_acc:.3f}')
print(f'Log-loss         : {ll:.4f}')
print(f'Draw recall      : {draw_recall:.3f}')
print(f'Draw precision   : {draw_prec:.3f}')
print()
print('Confusion matrix (rows=truth, cols=predicted):')
print('           guess_away  guess_draw  guess_home')
for i, n in enumerate(names):
    print(f'  true_{n}  ' + ''.join(f'{cm[i][j]:11d}' for j in range(3)))
print()
print(classification_report(yte, test_pred, target_names=names,
                             digits=3, zero_division=0))


## Cell 9 — Save + log

In [ ]:
torch.save({
    'model_state' : model.state_dict(),
    'feature_cols': FEATURE_COLS,
    'arch'        : {'n_features': N_FEATURES, 'n_classes': N_CLASSES,
                     'h1': 40, 'h2': 20},
    'temperature' : T_best,
    'warm_started': bool(TRANSFER_DONE),
    'strength_source': 'dixon_coles_v4',
}, MODELS_DIR / 'football_v4.pth')

with open(MODELS_DIR / 'scaler_v4.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved v4_backend/models/football_v4.pth')
print('Saved v4_backend/models/scaler_v4.pkl')

stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')
entry = f"""
## V4.2 Phase 3 Step 2 — neural net trained ({stamp})
- Architecture: {N_FEATURES}→40→20→{N_CLASSES}  ({n_params:,} params)
- Strength features: Dixon-Coles alpha/beta (replaced FIFA rank)
- Warm-started: {TRANSFER_DONE}
- Temperature T: {T_best:.3f}
- Test accuracy: {overall_acc:.3f}  (baseline {baseline_acc:.3f})
- Test log-loss: {ll:.4f}
- Draw recall: {draw_recall:.3f}  |  Draw precision: {draw_prec:.3f}
- Saved: v4_backend/models/football_v4.pth
"""
rp = RESULTS / 'RESULTS.md'
if not rp.exists(): rp.write_text('# V4.2 Results Log\n')
with rp.open('a') as f: f.write(entry)

import pandas as pd
row = pd.DataFrame([{{
    'timestamp': stamp, 'phase': 'phase3_train_v4',
    'test_acc': round(overall_acc, 4), 'baseline_acc': round(baseline_acc, 4),
    'test_logloss': round(ll, 4), 'draw_recall': round(draw_recall, 4),
    'temperature': round(T_best, 3),
}}])
mc = RESULTS / 'metrics.csv'
row.to_csv(mc, mode='a', header=not mc.exists(), index=False)

print(entry)
print('Phase 3 Step 2 complete.')
print()
print('Next: run validate_v4_neural.py to test on the 2425 holdout.')
